# Instrucciones

Baja el dataus dataset con el siguiente código: 

Este dataset es de empresas de Estados Unidos con datos anuales. La estructura del dataset es panel-data ya que cada empresa tiene varios periodos de información.
Tienes que correr un modelo de regresión para examinar qué tanto el earnings per share (epsp) influencía el retorno anual logarítmico considerando la industria a la que pertenece, y diseña el modelo para examinar si el efecto de epsp cambia dependiendo de la industria a la que pertenece la empresa.
El retorno anual logarítmico calcúlalo utilizando el stockprice (que es anual); recuerda que el dataset está en panel data.
Corre el modelo, imprime los resultados de regresión y realiza una interpretación detallada del modelo. En la interpretación explica en detalle los coeficientes, signo, magnitud y significancia estadística. Asegúrate que respondes la pregunta si el efecto de epsp en el retorno puede cambiar dependiendo de la industria.
(extra puntos) Utiliza al menos 1 medida para identificar outliers, identifica outliers y corre el mismo modelo y comenta si hubo alguna diferencia.  
Tienes que escribir con MAYÚSCULAS los pasos que sigues para correr el modelo, y
responder las preguntas de INTERPRETACIÓN CON TUS PALABRAS
SÓLO DEJA EL CÓDIGO NECESARIO!! (Extra código será penalizado)
SUBE tu archivo como "TuNombre.ipynb" 

IA utilizada para codigos: Claude Sonnet 4.5

In [1]:
import pandas as pd
import requests

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

url = 'https://www.apradie.com/datos/dataus.csv'
response = requests.get(url, headers=headers)
with open('dataus.csv', 'wb') as f:
    f.write(response.content)

dataus = pd.read_csv('dataus.csv')
display(dataus.head())

,firm,year,revenue,ebit,netincome,totalassets,totalliabilities,shortdebt,longdebt,retainedearnings,stockprice,marketvalue,epsp,industry
0,AAP,2001,2517639.0,98212.0,25005.0,1950615.0,1662044.0,58463.0,932022.0,-205294.0,14.405094,0.000,NaN,Trade
1,AAP,2002,3287883.0,201985.0,75967.0,1965225.0,1496869.0,11559.0,724832.0,-140275.0,14.158977,1745366.331,0.043525,Trade
2,AAP,2003,3493696.0,288234.0,172234.0,1983071.0,1351827.0,53305.0,422780.0,-15340.0,23.569340,3004390.809,0.057327,Trade
3,AAP,2004,3770297.0,328758.0,190968.0,2201962.0,1479647.0,51884.0,438300.0,172648.0,25.295056,3211784.154,0.059459,Trade
4,AAP,2005,4264971.0,408492.0,231910.0,2542149.0,1622378.0,82930.0,406040.0,407373.0,37.751481,4701384.545,0.049328,Trade


## EXPLORACIÓN INICIAL DE LOS DATOS

In [12]:
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats

print(dataus.shape)
print(dataus.columns.tolist())
display(dataus.head())
print(dataus.info())

(5975, 14)
['firm', 'year', 'revenue', 'ebit', 'netincome', 'totalassets', 'totalliabilities', 'shortdebt', 'longdebt', 'retainedearnings', 'stockprice', 'marketvalue', 'epsp', 'industry']


,firm,year,revenue,ebit,netincome,totalassets,totalliabilities,shortdebt,longdebt,retainedearnings,stockprice,marketvalue,epsp,industry
0,AAP,2001,2517639.0,98212.0,25005.0,1950615.0,1662044.0,58463.0,932022.0,-205294.0,14.405094,0.000,NaN,Trade
1,AAP,2002,3287883.0,201985.0,75967.0,1965225.0,1496869.0,11559.0,724832.0,-140275.0,14.158977,1745366.331,0.043525,Trade
2,AAP,2003,3493696.0,288234.0,172234.0,1983071.0,1351827.0,53305.0,422780.0,-15340.0,23.569340,3004390.809,0.057327,Trade
3,AAP,2004,3770297.0,328758.0,190968.0,2201962.0,1479647.0,51884.0,438300.0,172648.0,25.295056,3211784.154,0.059459,Trade
4,AAP,2005,4264971.0,408492.0,231910.0,2542149.0,1622378.0,82930.0,406040.0,407373.0,37.751481,4701384.545,0.049328,Trade


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5975 entries, 0 to 5974
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   firm              5975 non-null   object 
 1   year              5975 non-null   int64  
 2   revenue           5975 non-null   float64
 3   ebit              5975 non-null   float64
 4   netincome         5975 non-null   float64
 5   totalassets       5974 non-null   float64
 6   totalliabilities  5974 non-null   float64
 7   shortdebt         5971 non-null   float64
 8   longdebt          5972 non-null   float64
 9   retainedearnings  5934 non-null   float64
 10  stockprice        5848 non-null   float64
 11  marketvalue       5975 non-null   float64
 12  epsp              5818 non-null   float64
 13  industry          5975 non-null   object 
dtypes: float64(11), int64(1), object(2)
memory usage: 653.6+ KB
None


## CÁLCULO DEL RETORNO ANUAL LOGARÍTMICO

In [13]:
dataus = dataus.sort_values(['firm', 'year'])
dataus['log_return'] = dataus.groupby('firm')['stockprice'].transform(lambda x: np.log(x / x.shift(1)))
dataus_clean = dataus.dropna(subset=['log_return', 'epsp', 'industry'])
print(f"Observaciones originales: {len(dataus)}")
print(f"Observaciones después de limpiar: {len(dataus_clean)}")

Observaciones originales: 5975
Observaciones después de limpiar: 5349


## MODELO DE REGRESIÓN CON INTERACCIÓN INDUSTRIA-EPSP

In [14]:
model = smf.ols('log_return ~ epsp * C(industry)', data=dataus_clean).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             log_return   R-squared:                       0.161
Model:                            OLS   Adj. R-squared:                  0.160
Method:                 Least Squares   F-statistic:                     204.7
Date:                Thu, 16 Oct 2025   Prob (F-statistic):          2.86e-200
Time:                        11:45:52   Log-Likelihood:                -3717.1
No. Observations:                5349   AIC:                             7446.
Df Residuals:                    5343   BIC:                             7486.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

## INTERPRETACIÓN DEL MODELO

EL MODELO PRESENTA UN R² AJUSTADO DE 0.160, INDICANDO QUE APROXIMADAMENTE EL 16% DE LA VARIABILIDAD EN LOS RETORNOS LOGARÍTMICOS ES EXPLICADA POR LAS GANANCIAS POR ACCIÓN Y LA INDUSTRIA. EL MODELO ES ESTADÍSTICAMENTE SIGNIFICATIVO CON UN F-STATISTIC DE 204.7 Y P-VALOR CERCANO A CERO.

EL INTERCEPTO DE 0.0679 REPRESENTA EL RETORNO LOGARÍTMICO ESPERADO PARA LA INDUSTRIA BASE CUANDO EPSP ES CERO, SIENDO ESTADÍSTICAMENTE SIGNIFICATIVO. EL COEFICIENTE PRINCIPAL DE EPSP ES 0.5389 Y ALTAMENTE SIGNIFICATIVO, INDICANDO QUE EN LA INDUSTRIA DE REFERENCIA, UN AUMENTO DE UNA UNIDAD EN LAS GANANCIAS POR ACCIÓN SE ASOCIA CON UN INCREMENTO DE 0.5389 EN EL RETORNO LOGARÍTMICO.

LAS VARIABLES DE INDUSTRIA SOFTWARE & DATA Y TRADE NO MUESTRAN DIFERENCIAS SIGNIFICATIVAS EN EL INTERCEPTO COMPARADAS CON LA INDUSTRIA BASE, CON COEFICIENTES DE -0.0051 Y -0.0091 RESPECTIVAMENTE Y P-VALORES MAYORES A 0.05.

RESPECTO A LOS TÉRMINOS DE INTERACCIÓN, EL COEFICIENTE EPSP:C(INDUSTRY)[T.SOFTWARE & DATA] ES -0.0296 CON P-VALOR DE 0.610, NO SIGNIFICATIVO ESTADÍSTICAMENTE. ESTO INDICA QUE EL EFECTO DE EPSP EN SOFTWARE & DATA NO DIFIERE SIGNIFICATIVAMENTE DE LA INDUSTRIA BASE. SIN EMBARGO, EL TÉRMINO EPSP:C(INDUSTRY)[T.TRADE] ES -0.3423 Y ALTAMENTE SIGNIFICATIVO CON P-VALOR CERCANO A CERO. ESTO SIGNIFICA QUE EN LA INDUSTRIA TRADE, EL EFECTO TOTAL DE EPSP SOBRE EL RETORNO ES 0.5389 - 0.3423 = 0.1966, SUSTANCIALMENTE MENOR QUE EN LA INDUSTRIA BASE.

EN CONCLUSIÓN, SÍ, EL EFECTO DE EPSP SOBRE EL RETORNO LOGARÍTMICO CAMBIA DEPENDIENDO DE LA INDUSTRIA. ESPECÍFICAMENTE, EL IMPACTO POSITIVO DE LAS GANANCIAS POR ACCIÓN SOBRE LOS RETORNOS ES SIGNIFICATIVAMENTE MENOR EN LA INDUSTRIA TRADE COMPARADO CON LA INDUSTRIA DE REFERENCIA, MIENTRAS QUE EN SOFTWARE & DATA NO HAY DIFERENCIA SIGNIFICATIVA.

## IDENTIFICACIÓN DE OUTLIERS MEDIANTE Z-SCORE

In [ ]:
dataus_clean = dataus_clean.copy()
dataus_clean['z_log_return'] = np.abs(stats.zscore(dataus_clean['log_return']))
dataus_clean['z_epsp'] = np.abs(stats.zscore(dataus_clean['epsp']))
outliers = (dataus_clean['z_log_return'] > 3) | (dataus_clean['z_epsp'] > 3)
print(f"Outliers detectados: {outliers.sum()}")
dataus_no_outliers = dataus_clean[~outliers]
print(f"Observaciones sin outliers: {len(dataus_no_outliers)}")

## MODELO DE REGRESIÓN SIN OUTLIERS

In [19]:
model_no_outliers = smf.ols('log_return ~ epsp * C(industry)', data=dataus_no_outliers).fit()
print(model_no_outliers.summary())

                            OLS Regression Results                            
Dep. Variable:             log_return   R-squared:                       0.071
Model:                            OLS   Adj. R-squared:                  0.070
Method:                 Least Squares   F-statistic:                     80.43
Date:                Thu, 16 Oct 2025   Prob (F-statistic):           1.40e-81
Time:                        11:52:27   Log-Likelihood:                -2914.0
No. Observations:                5248   AIC:                             5840.
Df Residuals:                    5242   BIC:                             5879.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

## COMPARACIÓN DE RESULTADOS CON Y SIN OUTLIERS

AL ELIMINAR 101 OUTLIERS DETECTADOS MEDIANTE EL MÉTODO Z-SCORE, EL MODELO SIN OUTLIERS MUESTRA CAMBIOS SUSTANCIALES EN LOS RESULTADOS. EL R² AJUSTADO DISMINUYÓ DE 0.160 A 0.070, INDICANDO QUE LOS VALORES EXTREMOS ESTABAN INFLANDO ARTIFICIALMENTE EL PODER EXPLICATIVO DEL MODELO ORIGINAL.

EL COEFICIENTE PRINCIPAL DE EPSP AUMENTÓ DE 0.5389 A 0.6087, MANTENIÉNDOSE ALTAMENTE SIGNIFICATIVO. ESTO SUGIERE QUE EN DATOS TÍPICOS, EL EFECTO DE LAS GANANCIAS POR ACCIÓN SOBRE LOS RETORNOS ES AÚN MÁS FUERTE QUE LO ESTIMADO ORIGINALMENTE.

EL CAMBIO MÁS IMPORTANTE OCURRE EN LOS TÉRMINOS DE INTERACCIÓN. EL COEFICIENTE EPSP:C(INDUSTRY)[T.TRADE] CAMBIÓ DRÁSTICAMENTE DE -0.3423 (ALTAMENTE SIGNIFICATIVO) A -0.0904 (P-VALOR = 0.495, NO SIGNIFICATIVO). ESTO INDICA QUE LA CONCLUSIÓN ORIGINAL SOBRE LA DIFERENCIA SIGNIFICATIVA EN EL EFECTO DE EPSP EN LA INDUSTRIA TRADE ESTABA SESGADA POR VALORES ATÍPICOS. SIN LOS OUTLIERS, NO EXISTE EVIDENCIA ESTADÍSTICA DE QUE EL EFECTO DE EPSP DIFIERA ENTRE TRADE Y LA INDUSTRIA BASE.

SIMILARMENTE, EL TÉRMINO EPSP:C(INDUSTRY)[T.SOFTWARE & DATA] CAMBIÓ DE -0.0296 A 0.1450, PERO SIGUE SIENDO NO SIGNIFICATIVO (P-VALOR = 0.282).

EN CONCLUSIÓN, DESPUÉS DE REMOVER OUTLIERS, NO HAY EVIDENCIA ESTADÍSTICA DE QUE EL EFECTO DE EPSP SOBRE EL RETORNO LOGARÍTMICO VARÍE SIGNIFICATIVAMENTE ENTRE INDUSTRIAS. LOS RESULTADOS ORIGINALES QUE MOSTRABAN UN EFECTO DIFERENCIAL EN TRADE FUERON IMPULSADOS POR OBSERVACIONES ATÍPICAS. EL MODELO SIN OUTLIERS ES MÁS ROBUSTO Y CONFIABLE, REVELANDO QUE EL IMPACTO POSITIVO DE LAS GANANCIAS POR ACCIÓN SOBRE LOS RETORNOS ES CONSISTENTE A TRAVÉS DE LAS INDUSTRIAS ANALIZADAS.